In [ ]:
%pip install -qU \
  "langchain>=0.3" \
  "langgraph>=0.2" \
  "langchain-google-genai>=2.0" \
  "google-genai>=1.0" \
  "langchain-mcp-adapters==0.2.1" \
  "nest_asyncio" \
  "fastmcp>=2.0.0"

import nest_asyncio
nest_asyncio.apply()

print("Dependencies installed successfully.")

### 2. Configurer la clé API Gemini
Ajoutez votre clé API dans les secrets de Colab (icône clé à gauche) sous le nom `GOOGLE_API_KEY`.

In [ ]:
import os
from google.colab import userdata

try:
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
    print("✅ Clé API configurée.")
except Exception as e:
    print("❌ Erreur : Assurez-vous d'avoir ajouté 'GOOGLE_API_KEY' dans les secrets de Colab.")

### 3. Vérifier Node.js et NPM

In [ ]:
!node --version
!npx --version

### 4. Définir le répertoire de travail et créer le serveur MCP personnalisé
Nous allons créer un serveur `custom_ops` avec des outils pour manipuler du texte.

In [ ]:
import os
from pathlib import Path
import textwrap

# Définir le répertoire de travail
WORKDIR = os.path.abspath("/content/workspace")
os.makedirs(WORKDIR, exist_ok=True)

# Création du serveur personnalisé
server_path = Path("/content/custom_mcp_server.py")
server_path.write_text(textwrap.dedent("""
from fastmcp import FastMCP
from typing import Dict, List

mcp = FastMCP(name="custom_ops")

@mcp.tool
def ping() -> str:
    \"\"\"Outil de vérification de santé.\"\"\"
    return "pong"

@mcp.tool
def summarize_lines(lines: List[str]) -> Dict[str, int]:
    \"\"\"Compte les lignes totales et non vides.\"\"\"
    total = len(lines)
    nonempty = sum(1 for l in lines if l.strip())
    return {"total_lines": total, "nonempty_lines": nonempty}

if __name__ == "__main__":
    mcp.run(transport="stdio")
"""), encoding="utf-8")

print(f"Serveur personnalisé écrit dans : {server_path}")
print(f"Répertoire de travail : {WORKDIR}")

### 5. Connexion aux serveurs MCP (Filesystem + Custom)
Nous allons maintenant configurer le client pour orchestrer le serveur de fichiers standard et notre serveur personnalisé.

In [ ]:
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient

# Configuration des connexions
mcp_connections = {
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", WORKDIR],
    },
    "custom_ops": {
        "transport": "stdio",
        "command": "python",
        "args": [str(server_path)],
    }
}

async def initialize_tools():
    client = MultiServerMCPClient(mcp_connections, tool_name_prefix=True)
    # Nous récupérons les outils de manière asynchrone
    tools = await client.get_tools()
    return client, tools

# Initialisation
client, tools = asyncio.run(initialize_tools())

print(f"Nombre d'outils chargés : {len(tools)}")
print("Liste des outils :", [t.name for t in tools])

### 6. Créer l'Agent Gemini Orchestrateur
Nous utilisons `langgraph` pour créer un agent capable de décider quel outil appeler en fonction de votre demande.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

# Initialiser le modèle Gemini
model = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

# Créer l'agent avec les outils MCP collectés
agent_executor = create_react_agent(model, tools)

print("✅ Agent prêt à l'emploi.")

### 7. Test de l'Assistant (Scénario complet)
Demandons à l'agent de créer un fichier, d'y écrire du texte, puis d'utiliser notre outil personnalisé pour analyser le contenu.

In [ ]:
async def run_demo():
    query = """
    1. Crée un fichier nommé 'notes.txt' dans mon espace de travail.
    2. Écris 3 lignes de texte dedans (dont une ligne vide).
    3. Utilise l'outil custom_ops_summarize_lines pour analyser ce fichier.
    """

    async for event in agent_executor.astream(
        {"messages": [("user", query)]},
        stream_mode="values"
    ):
        message = event["messages"][-1]
        if hasattr(message, "content") and message.content:
            print(f"Agent: {message.content}\n")

# Exécution du test
await run_demo()

### 🎉 Projet Terminé !
Vous avez maintenant :
1. Un serveur MCP **Filesystem** (tierce partie).
2. Un serveur MCP **Custom** (Python FastMCP).
3. Un **Agent Gemini** qui orchestre les deux de manière dynamique via LangGraph.